# EmotionGAN + EmotionNet-CNN — Colab execution

Reproducible training of seven emotion-class GANs and a 146,759-parameter residual CNN on **FER-2013**. This notebook is a launcher for the modular Python package, not a separate algorithm fork. 

**Before running:** Upload/unzip the repository into `/content/EmotionGAN-EmotionNet-FER` and obtain `fer2013.csv` under its original access terms. Set runtime hardware to **GPU** if available. No data or trained results are bundled. The 97.89% paper claim is **not** treated as a target or predetermined outcome.

In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT = Path('/content/EmotionGAN-EmotionNet-FER')
if not PROJECT.exists():
    raise FileNotFoundError('Unzip the ZIP archive to /content/EmotionGAN-EmotionNet-FER before running.')
os.chdir(PROJECT)
print('Project:', Path.cwd())
!pip -q install -e .
!python -m emotionfer environment

## Supply the original FER-2013 dataset

Place the original `fer2013.csv` at `/content/fer2013.csv`, either via Colab's Files pane or by copying from Drive. The dataset must contain `emotion`, `pixels` (and `Usage` when running the official protocol). The project deliberately does not download or redistribute face photos automatically.

In [ ]:
from pathlib import Path
DATA = Path('/content/fer2013.csv')
if not DATA.exists():
    print('FER-2013 CSV is missing: upload it to /content/fer2013.csv using the left Files panel.')
else:
    print('Dataset:', DATA, 'Size (MB):', round(DATA.stat().st_size/1e6, 2))

## Choose an evaluation protocol

- `paper`: reconstructed 7,070-image subset, 72% train / 8% internal validation / 20% final test. Exact original row IDs unknown.
- `official`: FER-2013 `Training`/`PublicTest`/`PrivateTest` partitions, not numerically interchangeable with `paper`.
- `smoke`: one-epoch neural network test on synthetic *non-face* random images, not a scientific result.

Both paper and official configurations train seven GANs for 100 epochs and the classifier for 20 epochs.

In [ ]:
PROTOCOL = 'paper'  # 'paper' | 'official' | 'smoke'
RUN_DIR = Path('/content/fer_runs') / PROTOCOL
CONFIG = {'paper': 'configs/paper.yaml', 'official': 'configs/official.yaml',
          'smoke': 'configs/smoke.yaml'}[PROTOCOL]
if PROTOCOL == 'smoke':
    DATA = Path('/content/demo_fer2013.csv')
    subprocess.run([sys.executable, '-m', 'emotionfer', 'create-demo', '--out', str(DATA), '--per-class', '32'], check=True)
else:
    assert DATA.exists(), f'FER-2013 CSV not found: {DATA}'
cmd = [sys.executable, '-m', 'emotionfer', 'run', '--data', str(DATA),
       '--run-dir', str(RUN_DIR), '--config', CONFIG]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Inspect actual held-out results

These values come from your own test predictions and may differ from the manuscript.

In [ ]:
import json
from IPython.display import display, Image as DisplayImage
with open(RUN_DIR / 'evaluation' / 'metrics.json') as file:
    metric = json.load(file)
for key in ['n_test','accuracy','precision_macro','recall_macro','f1_macro','paper_harmonic_macro_f1']:
    print(f'{key}: {metric[key]}')
for name in ['class_balance.png','gan_losses.png']:
    fig = RUN_DIR / 'figures' / name
    if fig.exists(): display(DisplayImage(filename=str(fig), width=950))
for name in ['confusion_matrix.png', 'per_class_scores.png','roc_one_vs_rest.png']:
    display(DisplayImage(filename=str(RUN_DIR / 'evaluation' / name), width=850))

## Optional controlled comparisons

For added scientific rigor, run baselines and ablation variants (9 additional classifier trainings). The source manuscript does not disclose enough detail to reproduce the exact original five external baselines. These are explicitly identified, new controlled comparisons.

In [ ]:
# Uncomment to train full baseline + ablation suite on the same frozen split:
# subprocess.run([sys.executable, '-m', 'emotionfer', 'compare',
#                 '--run-dir', str(RUN_DIR), '--suite', 'all'], check=True)
print('Optional suite: python -m emotionfer compare --run-dir', RUN_DIR, '--suite all')

## Optional five-seed scientific report

Sequentially trains five independent GAN/CNN runs and aggregates actual test metrics. This is intentionally off by default due to significant compute requirements.

In [ ]:
# For real FER-2013 only. Uncomment when ready:
# subprocess.run([sys.executable, '-m', 'emotionfer', 'multi-seed',
#                 '--data', str(DATA), '--config', CONFIG,
#                 '--root', '/content/fer_runs/five_seeds',
#                 '--seeds', '42,123,2024,2025,3407'], check=True)
print('Multi-seed command documented in README.md')

In [ ]:
# Optionally copy run artifacts to Google Drive; nothing is uploaded automatically.
print('Trained final classifier:', RUN_DIR / 'classifier' / 'classifier_final.keras')
print('Per-image predicted probabilities:', RUN_DIR / 'evaluation' / 'predictions.csv')
print('Class/provenance manifest:', RUN_DIR / 'split_manifest.json')